In [1]:
import os
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset, DatasetDict
from nltk.translate.bleu_score import sentence_bleu

In [2]:
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name, use_auth_token=os.getenv("HUGGING_FACE_TOKEN"))
model = MBartForConditionalGeneration.from_pretrained(model_name, use_auth_token=os.getenv("HUGGING_FACE_TOKEN"))
tokenizer.src_lang = "kby_Latn"
tokenizer.tgt_lang = "en_XX" 

C:\Users\MOPHE\PycharmProjects\Kilba\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [3]:
data = {
    "train": [
        {"kilba": "Nja yah Yesu avu Baitalami atə hə'i Yahudiya, aku bəji nda təl Hiridus ku dzəga təlkur ti.",
         "english": "Now when Jesus was born in Bethlehem of Judaea in the days of Herod the king, behold, there came wise men from the east to Jerusalem."},
        {
            "kilba": "ngə nda jawa, “Aman ngə zər ngə nja yah təl njir Yahuda nga? Aka a la'atə ea sasəlga nyi təwa zədər pəchi, na cha nə ea shili ka ea hətə nyi səli.”",
            "english": "Saying, Where is he that is born King of the Jews? for we have seen his star in the east, and are come to worship him."}
    ],
    "validation": [
        {"kilba": "Aman ngə zər ngə nja yah təl njir Yahuda nga?",
         "english": "Where is he that is born King of the Jews?"}
    ]
}

In [7]:
def preprocess_function(examples):
    inputs = [example["kilba"] for example in examples]
    targets = [example["english"] for example in examples]
    
    # First, force the source language
    tokenizer.src_lang = "kby_Latn"
    model_inputs = tokenizer(
        inputs, 
        max_length=128, 
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    
    # Then, force the target language
    tokenizer.tgt_lang = "en_XX"
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=128,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [5]:
train_dataset = Dataset.from_dict({
    "kilba": [item["kilba"] for item in data["train"]],
    "english": [item["english"] for item in data["train"]]
})
validation_dataset = Dataset.from_dict({
    "kilba": [item["kilba"] for item in data["validation"]],
    "english": [item["english"] for item in data["validation"]]
})

datasets = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset
})

In [8]:
# Tokenize datasets
tokenized_datasets = datasets.map(
    lambda x: preprocess_function([{key: x[key][i] for key in x} for i in range(len(x["kilba"]))]),
    batched=True
)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

C:\Users\MOPHE\PycharmProjects\Kilba\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:4126: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [9]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=10,
    num_train_epochs=10,
    predict_with_generate=True,
    logging_dir="./logs",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

C:\Users\MOPHE\PycharmProjects\Kilba\.venv\Lib\site-packages\transformers\training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,No log,10.718092
2,No log,10.531563
3,No log,10.368380
4,No log,10.225205
5,No log,10.108869
6,No log,10.014707
7,No log,9.939399
8,No log,9.882633
9,No log,9.845189
10,No log,9.826262


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}


TrainOutput(global_step=10, training_loss=8.65728302001953, metrics={'train_runtime': 695.8005, 'train_samples_per_second': 0.029, 'train_steps_per_second': 0.014, 'total_flos': 5417824419840.0, 'train_loss': 8.65728302001953, 'epoch': 10.0})

In [10]:
trainer.save_model("kilba-to-english")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}


In [13]:
def translate_kilba_to_english(sentence):
    # Set source language before tokenization
    tokenizer.src_lang = "kby_Latn"
    
    # Tokenize without src_lang parameter
    inputs = tokenizer(sentence, return_tensors="pt", max_length=128, truncation=True)
    
    # Generate with forced BOS token for English
    outputs = model.generate(**inputs, forced_bos_token_id=tokenizer.lang_code_to_id["en_XX"])
    translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated_text

In [19]:
kilba_sentence = "ngə nda jawa, “Aman ngə zər ngə nja yah təl njir Yahuda nga?"
translated_text = translate_kilba_to_english(kilba_sentence)
print("Translated Text:", translated_text)

Translated Text: And he said, Where is he that is born of the sons of Ammon? where is he that is born of the sons of Ammon?


In [17]:
def calculate_bleu(reference, candidate):
    reference_tokens = reference.split()
    candidate_tokens = candidate.split()
    return sentence_bleu([reference_tokens], candidate_tokens)

In [21]:
reference_sentence = "Saying, Where is he that is born King of the Jews?"
candidate_sentence = "And he said, Where is he that is born of the sons of Ammon?"

bleu_score = calculate_bleu(reference_sentence, candidate_sentence)
print("BLEU Score:", bleu_score)

BLEU score: 0.3934995962231127
